# Interactive Table Tennis Simulation

In [ ]:
import importlib.util
import sys
from pathlib import Path

_root = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'pyproject.toml').exists()), None)
_expected_python = _root / '.venv' / 'Scripts' / 'python.exe' if _root else None
if _root is None or not _expected_python.exists() or Path(sys.executable).resolve() != _expected_python.resolve() or importlib.util.find_spec('table_tennis') is None:
    raise RuntimeError("Kernel incorrecto. Ejecuta scripts/setup_environment.ps1 y selecciona 'Python (TableTennis)'.")

import ipywidgets as widgets
import numpy as np
from IPython.display import display, clear_output, Video
from table_tennis import InitialConditions, simulate
from table_tennis.benchmarks import direct as benchmark_direct
from table_tennis.constants import NET_HEIGHT, TABLE_HEIGHT, TABLE_WIDTH
from table_tennis.visualization import animate_simulation
from table_tennis.visualization.notebook_video import notebook_video_path

benchmark_cases = list(benchmark_direct.build_cases())
services = sorted({case.service for case in benchmark_cases})
depths = list(benchmark_direct.DEPTHS.keys())
lanes = list(benchmark_direct.LANES.keys())

service_dropdown = widgets.Dropdown(options=services, value='pendulum', description='Servicio')
depth_dropdown = widgets.Dropdown(options=depths, value='short', description='Profundidad')
lane_dropdown = widgets.Dropdown(options=lanes, value='forehand', description='Zona')

pos_x = widgets.FloatSlider(value=0, min=-500, max=500, step=10, description='pos_x')
pos_y = widgets.FloatSlider(value=TABLE_WIDTH*4/8, min=0, max=TABLE_WIDTH, step=10, description='pos_y')
pos_z = widgets.FloatSlider(value=TABLE_HEIGHT + 2*NET_HEIGHT, min=TABLE_HEIGHT, max=TABLE_HEIGHT+400, step=10, description='pos_z')
vel_x = widgets.FloatSlider(value=7000, min=-10000, max=10000, step=100, description='vel_x')
vel_y = widgets.FloatSlider(value=-3000, min=-10000, max=10000, step=100, description='vel_y')
vel_z = widgets.FloatSlider(value=-3000, min=-10000, max=10000, step=100, description='vel_z')
omega_x = widgets.FloatSlider(value=0, min=-100, max=100, step=1, description='spin_x')
omega_y = widgets.FloatSlider(value=0, min=-100, max=100, step=1, description='spin_y')
omega_z = widgets.FloatSlider(value=75, min=-100, max=100, step=1, description='spin_z')

run_btn = widgets.Button(description='Run')
save_btn = widgets.Button(description='Save MP4')
video_duration = widgets.FloatSlider(value=3, min=1, max=8, step=.5, description='Duración')
video_fps = widgets.IntSlider(value=30, min=15, max=60, step=5, description='FPS')
out = widgets.Output()


def selected_benchmark_case():
    for case in benchmark_cases:
        if (case.service, case.depth, case.lane) == (service_dropdown.value, depth_dropdown.value, lane_dropdown.value):
            return case
    raise ValueError('No existe esa combinación en el benchmark')


def apply_benchmark_preset(_=None):
    ic = selected_benchmark_case().initial_conditions
    pos_x.value, pos_y.value, pos_z.value = ic.pos
    vel_x.value, vel_y.value, vel_z.value = ic.vel
    omega_x.value, omega_y.value, omega_z.value = tuple(np.array(ic.omega) / (2 * np.pi))


for dropdown in (service_dropdown, depth_dropdown, lane_dropdown):
    dropdown.observe(apply_benchmark_preset, names='value')


def build_ic():
    return InitialConditions(
        pos=(pos_x.value, pos_y.value, pos_z.value),
        vel=(vel_x.value, vel_y.value, vel_z.value),
        omega=(omega_x.value*2*np.pi, omega_y.value*2*np.pi, omega_z.value*2*np.pi),
    )


def on_run(_):
    ic = build_ic()
    result = simulate(ic)
    with out:
        clear_output(wait=True)
        animate_simulation(result)


def on_save(_):
    ic = build_ic()
    save_btn.disabled = True
    try:
        result = simulate(ic, t_max=video_duration.value)
        path = notebook_video_path(
            '01_direct_trajectory_explorer',
            f'{service_dropdown.value}-{depth_dropdown.value}-{lane_dropdown.value}',
        )
        animate_simulation(result, save=str(path), fps=video_fps.value)
        with out:
            clear_output(wait=True)
            print(f'Video guardado en: {path.resolve()}')
            display(Video(filename=str(path), embed=True))
    finally:
        save_btn.disabled = False


run_btn.on_click(on_run)
save_btn.on_click(on_save)
apply_benchmark_preset()

ui = widgets.VBox([
    widgets.HBox([service_dropdown, depth_dropdown, lane_dropdown]),
    pos_x, pos_y, pos_z,
    vel_x, vel_y, vel_z,
    omega_x, omega_y, omega_z,
    widgets.HBox([video_duration, video_fps]),
    widgets.HBox([run_btn, save_btn]),
    out
])

display(ui)
